# 📈 Quantiles in Statistics & Data Science: SAT Score Analysis (Skeleton)

## Expanded Project: From Theory to Decision-Making + Machine Learning Applications

### Learning Objectives
- Master the theory and intuition behind **quantiles**, deciles, quartiles, and percentiles
- Use `numpy.quantile()` and `pandas.quantile()` effectively (including interpolation methods)
- Visualize distributions with quantile reference lines
- Interpret a specific value (SAT = 1350) within different distributions
- Make data-driven college application decisions (Safety / Match / Reach)
- Apply quantiles in **Machine Learning** workflows (preprocessing, quantile regression, robust scaling, feature engineering)
- Build an interactive simulation to test "what if my score was X?"

**The Scenario**  
You have SAT scores of accepted students from three fake universities. You scored **1350**.  
Using **deciles** (10-quantiles), determine:
- Which tenth of the accepted students you would be in at each school
- Which schools are realistic, matches, or reaches
- How sensitive the recommendation is to small changes in your score

---
**How to use this notebook**: Fill in the `# TODO` sections. Compare with the Solution notebook when needed.


## 1. Theory: Quantiles – The Backbone of Distribution Analysis

### What are Quantiles?
**Quantiles** are values that split a dataset into groups of **equal size**.

- If you have **n quantiles**, the data is divided into **n + 1** groups of equal size.
- The **median** is the only **2-quantile** (50th percentile). It splits the data into two equal halves.
- **Quartiles** (4-quantiles): Q1 (25th), Q2 (median), Q3 (75th). The Interquartile Range (IQR = Q3 - Q1) is a robust measure of spread.
- **Deciles** (10-quantiles): Split data into 10 equal groups (used in this project).
- **Percentiles** (100-quantiles): Split data into 100 groups. Very common in standardized testing (SAT, GRE, etc.).

### Why Quantiles Matter in Data Science
1. **Robustness to Outliers**: Unlike the mean, quantiles are not pulled by extreme values.
2. **Understanding Skewness**: If mean > median → right skew. Quantiles give a fuller picture.
3. **Percentile Ranks**: "I scored in the 85th percentile" is more informative than the raw score.
4. **Outlier Detection**: Values below Q1 - 1.5×IQR or above Q3 + 1.5×IQR are potential outliers.
5. **Feature Engineering**: Creating "percentile rank" or "decile bin" features often improves ML models.
6. **Quantile Regression**: Instead of predicting the mean, we can predict specific quantiles (very useful for costs, times, risks).

### Key Python Functions
- `np.quantile(data, q)` where `q` is a float or array of floats between 0 and 1.
- `np.percentile(data, q)` — same as above but `q` is in [0, 100].
- `pd.Series.quantile(q)` — supports different interpolation methods (`'linear'`, `'lower'`, `'higher'`, `'nearest'`, `'midpoint'`).

**Important**: For small datasets or discrete data, the exact value at a quantile can depend on the interpolation method.


## 2. Data Generation & Visualization with Deciles

We generate three realistic distributions of accepted SAT scores (different selectivity levels).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
np.random.seed(42)

# Generate realistic accepted student SAT distributions
n_students = 500
school_one   = np.random.normal(1250, 120, n_students).clip(400, 1600)   # Less selective
school_two   = np.random.normal(1380, 90, n_students).clip(400, 1600)    # Moderately selective
school_three = np.random.normal(1480, 70, n_students).clip(400, 1600)    # Highly selective

deciles_one   = np.quantile(school_one,   np.arange(0.1, 1.0, 0.1))
deciles_two   = np.quantile(school_two,   np.arange(0.1, 1.0, 0.1))
deciles_three = np.quantile(school_three, np.arange(0.1, 1.0, 0.1))

def plot_school(ax, data, deciles, title, target_score=1350):
    ax.hist(data, bins=30, color="#4FC3F7", edgecolor="#0277BD", alpha=0.7)
    for i, d in enumerate(deciles):
        ax.axvline(x=d, color="red", linestyle="--", alpha=0.7, linewidth=1.2)
        ax.text(d, ax.get_ylim()[1]*0.9, f"D{i+1}", rotation=90, va="top", ha="right", fontsize=8, color="darkred")
    ax.axvline(x=target_score, color="green", linestyle="-", linewidth=2.5, label=f"Your score: {target_score}")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("SAT Score")
    ax.set_ylabel("Number of Accepted Students")
    ax.legend(loc="upper left")

fig, axes = plt.subplots(3, 1, figsize=(10, 9))
plot_school(axes[0], school_one,   deciles_one,   "School One (Less Selective)")
plot_school(axes[1], school_two,   deciles_two,   "School Two (Moderately Selective)")
plot_school(axes[2], school_three, deciles_three, "School Three (Highly Selective)")
plt.tight_layout()
plt.show()

## 3. Interpreting Your Score (SAT = 1350)

### Step-by-step Analysis for Each School

**School One (Less Selective)**  
Your score of 1350 falls in a high decile. You would be in a strong position — likely **top 20-30%** of accepted students. This is generally a **Safety** school for you.

**School Two (Moderately Selective)**  
1350 is around the middle or slightly above. You would likely be in the **4th–6th decile**. This looks like a **Match** school.

**School Three (Highly Selective)**  
1350 is lower in the distribution. You would probably be in the **2nd or 3rd decile**. This would be a **Reach** school, and possibly unrealistic unless you have other strong factors (GPA, essays, extracurriculars, legacy, etc.).

### Recommendation Logic (Simple Rule of Thumb)
- **> 70th percentile** (above 7th decile) → Safety school
- **30th – 70th percentile** → Match school  
- **< 30th percentile** (below 3rd decile) → Reach / Unrealistic without exceptional other qualifications


## 4. 🎮 Simulation: Change Your Score or School Distributions

Modify the parameters below and re-run the cell to see how recommendations change.


In [ ]:
# ============== SIMULATION PARAMETERS (MODIFY THESE) ==============
YOUR_SCORE = 1350          # Try 1200, 1400, 1500, etc.
SCHOOL_ONE_MEAN = 1250
SCHOOL_TWO_MEAN = 1380
SCHOOL_THREE_MEAN = 1480
SCHOOL_SPREAD = 100        # Standard deviation (higher = more overlap)
N_STUDENTS = 500
# =====================================================================

np.random.seed(42)
s1 = np.random.normal(SCHOOL_ONE_MEAN, SCHOOL_SPREAD, N_STUDENTS).clip(400, 1600)
s2 = np.random.normal(SCHOOL_TWO_MEAN, SCHOOL_SPREAD, N_STUDENTS).clip(400, 1600)
s3 = np.random.normal(SCHOOL_THREE_MEAN, SCHOOL_SPREAD, N_STUDENTS).clip(400, 1600)

dec1 = np.quantile(s1, np.arange(0.1, 1.0, 0.1))
dec2 = np.quantile(s2, np.arange(0.1, 1.0, 0.1))
dec3 = np.quantile(s3, np.arange(0.1, 1.0, 0.1))

def get_decile(score, deciles):
    for i, d in enumerate(deciles):
        if score <= d:
            return i + 1
    return 10

print(f"Your SAT Score: {YOUR_SCORE}")
print(f"School One decile   : {get_decile(YOUR_SCORE, dec1)} → {'Safety' if get_decile(YOUR_SCORE, dec1) >= 7 else 'Match/Reach'}")
print(f"School Two decile   : {get_decile(YOUR_SCORE, dec2)} → {'Safety' if get_decile(YOUR_SCORE, dec2) >= 7 else 'Match/Reach'}")
print(f"School Three decile : {get_decile(YOUR_SCORE, dec3)} → {'Safety' if get_decile(YOUR_SCORE, dec3) >= 7 else 'Match/Reach'}")

# Quick re-plot of one school for visual feedback
plt.figure(figsize=(8,4))
plt.hist(s2, bins=30, color="#81C784", edgecolor="#2E7D32", alpha=0.7)
for d in dec2:
    plt.axvline(d, color="red", ls="--", alpha=0.6)
plt.axvline(YOUR_SCORE, color="green", lw=2.5, label=f"Your score = {YOUR_SCORE}")
plt.title(f"School Two with YOUR_SCORE = {YOUR_SCORE}")
plt.xlabel("SAT Score")
plt.legend()
plt.show()

## 5. Alternate Ways to Compute & Visualize Quantiles

**Pandas way** (very common in real workflows):
```python
df = pd.DataFrame({"School_One": school_one})
print(df["School_One"].quantile([0.1, 0.5, 0.9]))
```

**Seaborn + annotation** (cleaner labels):
Use `sns.histplot` + manual text annotations for deciles.

**Exact percentile rank** (more precise than decile):
```python
from scipy import stats
percentile_rank = stats.percentileofscore(school_two, 1350)
print(f"You are in the {percentile_rank:.1f}th percentile at School Two")
```


## 6. Machine Learning Applications of Quantiles

Quantiles are extremely useful beyond basic EDA:

1. **Robust Scaling / Outlier Handling**
   ```python
   from sklearn.preprocessing import RobustScaler
   # Uses IQR (based on quantiles) instead of mean/std
   ```

2. **Feature Engineering**
   - Create a new column: `decile_rank = pd.qcut(scores, 10, labels=False)`
   - Binning continuous features into quantiles often helps tree models.

3. **Quantile Regression** (predicting intervals, not just point estimates)
   - Useful for predicting house prices, delivery times, medical costs where you care about the 90th percentile (worst case) or 10th percentile.

4. **Anomaly Detection**
   - Flag values below 1st percentile or above 99th percentile as potential anomalies.

5. **Model Evaluation**
   - Pinball loss / quantile loss is used to evaluate probabilistic forecasts.

**Takeaway for Data Scientists**: Always compute quantiles during EDA, especially when the target variable or important features are skewed.


## 🗺️ Flowchart of the Quantile Analysis Process

```mermaid
flowchart TD
    A[Load or Generate Score Data] --> B[Compute Deciles using np.quantile]
    B --> C[Visualize Histogram + Vertical Decile Lines]
    C --> D[Locate Target Score (1350) within Deciles]
    D --> E[Interpret Position: Safety / Match / Reach]
    E --> F[Simulation: Change score or distribution parameters]
    F --> G[Apply ML Techniques: Robust scaling, quantile features, quantile regression]
    G --> H[Final Recommendation + Communication]
```


## ✏️ More Practice Exercises

1. Compute the **quartiles** (Q1, Q2, Q3) and **IQR** for all three schools. Which school has the largest spread in the middle 50% of accepted students?
2. Write a reusable function `get_decile_and_recommendation(score, data, school_name)` that returns both the decile and a recommendation string.
3. Create a **boxplot + swarmplot** overlay for the three schools using seaborn.
4. Add a fourth school with a very narrow distribution (highly selective with little variance) and re-analyze.
5. **ML Practice**: Create a new feature `decile` for each school using `pd.qcut`. Then imagine you are building a model to predict "likelihood of admission" — how could this feature help?
6. Research and implement a simple **quantile regression** example using `statsmodels` (optional advanced).


## ✅ Key Takeaways & Summary

- **Quantiles** give a robust, outlier-resistant view of distributions.
- **Deciles** are excellent for dividing data into 10 understandable groups.
- A single score (1350) can mean very different things depending on the selectivity of the school.
- **Simulation** helps you understand sensitivity: small changes in your score or school averages can move you across decision boundaries.
- In **Machine Learning**, quantiles power robust preprocessing, feature engineering, and advanced modeling techniques like quantile regression.

**Final Advice for College Applications**:
Use this kind of analysis, but remember that SAT is only one part of your application. Strong essays, recommendations, and fit can move the needle significantly, especially at reach schools.

This project demonstrates how fundamental statistical concepts (quantiles) directly support real-world decision making and modern data science workflows.
